<a href="https://colab.research.google.com/github/g25ait2032-prog/nagaananth/blob/main/weave/cookbooks/source/Intro_to_Weave_Hello_Trace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Traces

<img src="https://cdn.prod.website-files.com/62ba1fb86485b6d5029975c4/69b1b3d9c77f6e5294374be1_Endorsed_primary_goldblack.png" width="400" alt="Weights & Biases" />

Weave is a toolkit for developing AI-powered applications.

Use Weave traces to capture the inputs, outputs, and internal structure of your Python function automatically to observe and debug LLM applications.

When you decorate a function with `@weave.op`, Weave records a rich trace of the function while it runs, including any nested operations or external API calls. Use the trace to to debug, understand, and visualize interactions between your code and LLM models, without leaving your notebook.

To get started, complete the prerequisites. Then, define a function decorated with `@weave.op` decorator and run it on an example input to track LLM calls. Weave captures and visualizes the trace automatically.

In [1]:
# Ensure your dependencies are installed with:
!pip install --quiet jedi openai weave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 62.7 MB/s eta 0:00:00


In [2]:
import os
import getpass

#@title Set up your credentials
inference_provider = "W&B Inference" #@param ["W&B Inference", "OpenAI"]

# Set up your W&B project and credentials
os.environ["WANDB_ENTITY_PROJECT"] = input("Set up your W&B project (team name/project name): ")
os.environ["WANDB_API_KEY"] = getpass.getpass("Set up your W&B API key (Create an API key at https://wandb.ai/settings): ")

# Set up your OpenAI API key
if inference_provider == "OpenAI":
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key (Find it at https://platform.openai.com/api-keys): ")

Set up your W&B project (team name/project name): MLOps_Assg2
Set up your W&B API key (Create an API key at https://wandb.ai/settings): ··········


In [5]:
import os

# 1. Double check your entity/project.
# It should look like "nagaananth/huggingface"
os.environ["WANDB_ENTITY_PROJECT"] = "g25ait2032-iit-jodhpur/huggingface"

# 2. Re-initialize weave with the explicit project string
import weave
weave.init(os.environ["WANDB_ENTITY_PROJECT"])

# 3. Test the completion again
review = "The detective found the blood-stained glove near the fireplace."
print(classify_genre_llm(review))

Output()

weave: Logged in as Weights & Biases user: g25ait2032.
weave: View Weave data at https://wandb.ai/g25ait2032-iit-jodhpur/huggingface/weave
weave: 🍩 https://wandb.ai/g25ait2032-iit-jodhpur/huggingface/r/call/019e1090-b36a-7121-897f-21ac239f4a78


Genre: Mystery


In [6]:
test_reviews = [
    "A star-crossed romance set in the streets of Verona.",
    "The 1945 conference that changed the map of Europe forever.",
    "Space marines fighting aliens on a distant moon base."
]

print("Running Weave Traces...")
for r in test_reviews:
    prediction = classify_genre_llm(r)
    print(f"Review: {r[:30]}... -> Predicted: {prediction}")

weave: 🍩 https://wandb.ai/g25ait2032-iit-jodhpur/huggingface/r/call/019e1091-6a01-7316-b720-fee6a8e30202


Running Weave Traces...


weave: 🍩 https://wandb.ai/g25ait2032-iit-jodhpur/huggingface/r/call/019e1091-6b3b-71a0-9422-56c27b82356c


Review: A star-crossed romance set in ... -> Predicted: Romance


weave: 🍩 https://wandb.ai/g25ait2032-iit-jodhpur/huggingface/r/call/019e1091-6c49-7a4b-b32a-75806ed4766b


Review: The 1945 conference that chang... -> Predicted: Genre: History
Review: Space marines fighting aliens ... -> Predicted: Genre: **Fantasy** (or **Science Fiction**, though it's not listed here — the closest match from the given options is **Fantasy**). 

Note: While "Space marines fighting aliens on a distant moon base" is more characteristic of **Science Fiction**, if **Science Fiction** is not an available option, **Fantasy** is the best fit among the provided genres. However, if **Science Fiction** were an option, it would be more accurate.


In [7]:
import pandas as pd

# The reviews that gave your DistilBERT model trouble earlier
test_reviews = [
    "The detective found a single fingerprint on the cold steel of the safe.",
    "The 19th-century industrial revolution changed the world forever.",
    "A collection of haikus about the changing seasons in Japan."
]

results = []

for text in test_reviews:
    # 1. Get prediction from your Hugging Face Model (DistilBERT)
    # (Assuming 'classifier' is your DistilBERT pipeline from earlier)
    distil_res = classifier(text)[0]['label']

    # 2. Get prediction from the LLM (Traced via Weave)
    llm_res = classify_genre_llm(text)

    results.append({
        "Review Snapshot": text[:40] + "...",
        "DistilBERT (v2)": distil_res,
        "LLM (Qwen via Weave)": llm_res.replace("Genre: ", "").strip()
    })

# Display as a professional table
df_cmp = pd.DataFrame(results)
print(df_cmp.to_markdown())

NameError: name 'classifier' is not defined

In [3]:
from openai import OpenAI
import weave

weave.init(os.environ["WANDB_ENTITY_PROJECT"])

@weave.op  # Decorator to track requests
def create_completion(message: str) -> str:
    if inference_provider == "W&B Inference":
      client = OpenAI(
          base_url="https://api.inference.wandb.ai/v1",
          api_key=os.environ["WANDB_API_KEY"],
          project=os.environ["WANDB_ENTITY_PROJECT"],
      )
      model_name: str = "OpenPipe/Qwen3-14B-Instruct"
    if inference_provider == "OpenAI":
      client = OpenAI()
      model_name: str = "gpt-4.1-nano"
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": message},
        ],
    )
    return response.choices[0].message.content


message = "Tell me a joke."
create_completion(message)

weave: Logged in as Weights & Biases user: g25ait2032.
weave: View Weave data at https://wandb.ai/g25ait2032-iit-jodhpur/MLOps_Assg2/weave


BadRequestError: Error code: 400 - {'error': {'message': 'Invalid Authentication', 'type': 'invalid_request_error'}}